# 平均余命データの分析

世界の平均余命を探求する2つのデータセット:
- **Gapminder** (1952-2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO 平均余命データ** (2000-2015): 193か国、22の指標（死亡率、BMI、GDP、就学年数など）

このワークブックでは、**Python** と **R** の両方で CSV のインポートと分析を行う様子を実演します。

## 1. セットアップ: パッケージのインストールとデータセットのダウンロード

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly をインストールしました')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"既に存在します: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"ダウンロード完了 {name}: {lines} 行")

## 2. Gapminder: Python による探索的データ分析

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"形状: {gap.shape}")
print(f"大陸: {sorted(gap['continent'].unique())}")
print(f"年の範囲: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# 大陸別の平均余命の推移
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='大陸別の平均余命 (1952-2007年)',
              labels={'lifeExp': '平均余命（年）', 'year': '年'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# GDP vs 平均余命 (2007年)、バブルサイズ = 人口
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='GDP vs 平均余命 (2007年)',
                 labels={'gdpPercap': '1人当たり GDP（対数）', 'lifeExp': '平均余命'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: R による探索的データ分析

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# 大陸別の平均余命の分布（箱ひげ図）
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "大陸別の平均余命",
        xlab = "大陸", ylab = "平均余命（年）",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# 平均余命の改善上位10か国 (1952年 vs 2007年)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "上位10か国: 平均余命の伸び (1952-2007年)",
        xlab = "伸びた年数",
        col = "#00CC96", border = NA)

## 4. WHO 平均余命データ: Python による探索的データ分析

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"形状: {who.shape}")
print(f"列: {list(who.columns)}")
print(f"\n欠損値（上位5件）:")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# 開発途上国 vs 先進国: 事前ビン分割された平均余命の分布
# 明示的な棒グラフ座標により、ブラウザの Plotly ブリッジ経由でも一貫して描画されます。
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='平均余命: 開発途上国 vs 先進国',
             labels={'Life expectancy': '平均余命（年）'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# 就学年数 vs 平均余命
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='就学年数 vs 平均余命 (2014年)',
                 labels={'Life expectancy': '平均余命（年）',
                         'Schooling': '就学年数'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO 平均余命データ: R による探索的データ分析

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\n国数:", length(unique(who$Country)))
cat("\n年の範囲:", range(who$Year))

In [ ]:
# 相関: 成人死亡率 vs 平均余命
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "成人死亡率 vs 平均余命",
     xlab = "成人死亡率（1000人当たり）",
     ylab = "平均余命（年）")
legend("topright", legend = c("先進国", "開発途上国"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# 単純線形モデル: 平均余命の予測因子は何か？
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## 主な知見

- 世界の平均余命は全体として伸びていますが、大陸間には依然として大きな格差が存在します
- GDP と就学年数は、平均余命に対する強い正の予測因子です
- 成人死亡率は最も強い負の予測因子です
- 開発途上国では、結果（平均余命）のばらつきがはるかに大きくなっています